In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              roc_curve, precision_recall_curve, average_precision_score,
                              f1_score, accuracy_score)
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ─────────────────────────────────────────────
# PALETTE
# ─────────────────────────────────────────────
BG       = "#0d0f1a"
PANEL    = "#131625"
ACCENT1  = "#00f5c4"   # teal
ACCENT2  = "#ff4d6d"   # red
ACCENT3  = "#7b61ff"   # purple
ACCENT4  = "#ffd166"   # gold
TEXT     = "#e8eaf6"
MUTED    = "#5c6080"

plt.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor':   PANEL,
    'axes.edgecolor':   MUTED,
    'axes.labelcolor':  TEXT,
    'xtick.color':      MUTED,
    'ytick.color':      MUTED,
    'text.color':       TEXT,
    'grid.color':       '#1e2140',
    'grid.linestyle':   '--',
    'grid.alpha':       0.6,
    'font.family':      'monospace',
})

print("=" * 60)
print("   CUSTOMER CHURN PREDICTION — FULL ML PIPELINE")
print("=" * 60)


   CUSTOMER CHURN PREDICTION — FULL ML PIPELINE


In [4]:
# ─────────────────────────────────────────────
# 1. LOAD & CLEAN
# ─────────────────────────────────────────────
df = pd.read_csv('Churn_Modelling.csv')
df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df = pd.get_dummies(df, columns=['Geography'], drop_first=False)

print(f"\n✔  Dataset loaded  |  rows={len(df):,}  |  churn_rate={df['Exited'].mean():.1%}")


✔  Dataset loaded  |  rows=10,000  |  churn_rate=20.4%


In [5]:
# ─────────────────────────────────────────────
# 2. FEATURE ENGINEERING
# ─────────────────────────────────────────────
df['BalanceSalaryRatio']   = df['Balance'] / (df['EstimatedSalary'] + 1)
df['TenureByAge']          = df['Tenure']  / (df['Age'] + 1)
df['CreditScorePerAge']    = df['CreditScore'] / df['Age']
df['ProductsPerTenure']    = df['NumOfProducts'] / (df['Tenure'] + 1)
df['IsHighValue']          = ((df['Balance'] > df['Balance'].quantile(0.75)) & (df['IsActiveMember'] == 1)).astype(int)
df['AgeGroup']             = pd.cut(df['Age'], bins=[0,30,45,60,200], labels=[0,1,2,3]).astype(int)
df['ZeroBalance']          = (df['Balance'] == 0).astype(int)

print(f"✔  Feature engineering done  |  total_features={df.shape[1]-1}")

✔  Feature engineering done  |  total_features=19


In [6]:
# ─────────────────────────────────────────────
# 3. SPLIT + SMOTE
# ─────────────────────────────────────────────
X = df.drop('Exited', axis=1)
y = df['Exited']
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_res_s  = scaler.fit_transform(X_res)
X_test_s = scaler.transform(X_test)
X_train_s = scaler.transform(X_train)

print(f"✔  SMOTE balanced  |  before={y_train.value_counts().to_dict()}  |  after={pd.Series(y_res).value_counts().to_dict()}")

✔  SMOTE balanced  |  before={0: 6370, 1: 1630}  |  after={1: 6370, 0: 6370}


In [7]:
# ─────────────────────────────────────────────
# 4. TRAIN MODELS
# ─────────────────────────────────────────────
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=0.5, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=300, max_depth=10,
                                                   min_samples_leaf=5, random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                       max_depth=4, subsample=0.8, random_state=42),
    "XGBoost":             xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                                              subsample=0.8, colsample_bytree=0.8,
                                              use_label_encoder=False, eval_metric='logloss',
                                              random_state=42, n_jobs=-1),
}

results   = {}
proba_dict = {}

for name, model in models.items():
    X_tr = X_res_s if name == "Logistic Regression" else X_res
    X_te = X_test_s if name == "Logistic Regression" else X_test

    model.fit(X_tr, y_res)
    preds = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]

    results[name] = {
        "Accuracy":  accuracy_score(y_test, preds),
        "F1":        f1_score(y_test, preds),
        "ROC-AUC":   roc_auc_score(y_test, proba),
        "PR-AUC":    average_precision_score(y_test, proba),
        "preds":     preds,
        "proba":     proba,
    }
    proba_dict[name] = proba
    print(f"  [{name:22s}]  AUC={results[name]['ROC-AUC']:.4f}  F1={results[name]['F1']:.4f}")

# Ensemble
ensemble = VotingClassifier(
    estimators=[('rf', models['Random Forest']),
                ('gb', models['Gradient Boosting']),
                ('xgb', models['XGBoost'])],
    voting='soft')
ensemble.fit(X_res, y_res)
ens_preds = ensemble.predict(X_test)
ens_proba = ensemble.predict_proba(X_test)[:, 1]
results["Ensemble"] = {
    "Accuracy": accuracy_score(y_test, ens_preds),
    "F1":       f1_score(y_test, ens_preds),
    "ROC-AUC":  roc_auc_score(y_test, ens_proba),
    "PR-AUC":   average_precision_score(y_test, ens_proba),
    "preds":    ens_preds,
    "proba":    ens_proba,
}
proba_dict["Ensemble"] = ens_proba
print(f"  [{'Ensemble':22s}]  AUC={results['Ensemble']['ROC-AUC']:.4f}  F1={results['Ensemble']['F1']:.4f}")

best_name = max(results, key=lambda k: results[k]['ROC-AUC'])
print(f"\n★  Best model → {best_name}  (AUC={results[best_name]['ROC-AUC']:.4f})")


  [Logistic Regression   ]  AUC=0.7559  F1=0.4709
  [Random Forest         ]  AUC=0.8510  F1=0.5794
  [Gradient Boosting     ]  AUC=0.8677  F1=0.6319
  [XGBoost               ]  AUC=0.8615  F1=0.6262
  [Ensemble              ]  AUC=0.8656  F1=0.6211

★  Best model → Gradient Boosting  (AUC=0.8677)


In [9]:
# ─────────────────────────────────────────────
# 5. MEGA DASHBOARD  (3 rows × 4 cols)
# ─────────────────────────────────────────────
fig = plt.figure(figsize=(26, 22), facecolor=BG)
fig.suptitle("CUSTOMER CHURN PREDICTION — MODEL INTELLIGENCE DASHBOARD",
             fontsize=18, fontweight='bold', color=ACCENT1,
             y=0.98, fontfamily='monospace')

gs = GridSpec(3, 4, figure=fig, hspace=0.52, wspace=0.38,
              left=0.05, right=0.97, top=0.94, bottom=0.04)

COLORS = [ACCENT1, ACCENT2, ACCENT3, ACCENT4, "#60a5fa"]
model_names = list(results.keys())

# ── 5.1  ROC Curves (row0, col0-1) ──────────────────────────────
ax_roc = fig.add_subplot(gs[0, :2])
ax_roc.plot([0,1],[0,1], color=MUTED, lw=1, ls='--')
for i, (name, res) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    ax_roc.plot(fpr, tpr, color=COLORS[i], lw=2.2,
                label=f"{name}  (AUC={res['ROC-AUC']:.3f})")
ax_roc.set_title("ROC CURVES", color=ACCENT1, fontsize=12, fontweight='bold', pad=10)
ax_roc.set_xlabel("False Positive Rate"); ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend(loc='lower right', fontsize=8, framealpha=0.2, labelcolor=TEXT)
ax_roc.grid(True)

# ── 5.2  PR Curves (row0, col2-3) ────────────────────────────────
ax_pr = fig.add_subplot(gs[0, 2:])
for i, (name, res) in enumerate(results.items()):
    prec, rec, _ = precision_recall_curve(y_test, res['proba'])
    ax_pr.plot(rec, prec, color=COLORS[i], lw=2.2,
               label=f"{name}  (AP={res['PR-AUC']:.3f})")
ax_pr.set_title("PRECISION-RECALL CURVES", color=ACCENT1, fontsize=12, fontweight='bold', pad=10)
ax_pr.set_xlabel("Recall"); ax_pr.set_ylabel("Precision")
ax_pr.legend(loc='upper right', fontsize=8, framealpha=0.2, labelcolor=TEXT)
ax_pr.grid(True)

# ── 5.3  Model Comparison Bar (row1, col0) ───────────────────────
ax_bar = fig.add_subplot(gs[1, 0])
metrics = ["Accuracy", "F1", "ROC-AUC", "PR-AUC"]
x = np.arange(len(metrics))
bar_w = 0.15
for i, name in enumerate(model_names):
    vals = [results[name][m] for m in metrics]
    bars = ax_bar.bar(x + i*bar_w, vals, bar_w, label=name, color=COLORS[i], alpha=0.88)
ax_bar.set_xticks(x + bar_w*2)
ax_bar.set_xticklabels(metrics, fontsize=8)
ax_bar.set_ylim(0.5, 1.0)
ax_bar.set_title("MODEL COMPARISON", color=ACCENT1, fontsize=11, fontweight='bold', pad=10)
ax_bar.legend(fontsize=6.5, framealpha=0.2, labelcolor=TEXT)
ax_bar.grid(True, axis='y')

# ── 5.4  Confusion Matrix — best model (row1, col1) ──────────────
ax_cm = fig.add_subplot(gs[1, 1])
cm = confusion_matrix(y_test, results[best_name]['preds'])
im = ax_cm.imshow(cm, cmap='YlOrRd', aspect='auto')
for i in range(2):
    for j in range(2):
        ax_cm.text(j, i, f"{cm[i,j]}", ha='center', va='center',
                   fontsize=16, fontweight='bold',
                   color='white' if cm[i,j] > cm.max()/2 else 'black')
ax_cm.set_xticks([0,1]); ax_cm.set_yticks([0,1])
ax_cm.set_xticklabels(['Predicted\nRetained','Predicted\nChurned'], fontsize=8)
ax_cm.set_yticklabels(['Actual\nRetained','Actual\nChurned'], fontsize=8)
ax_cm.set_title(f"CONFUSION MATRIX\n({best_name})", color=ACCENT1, fontsize=11, fontweight='bold', pad=10)

# ── 5.5  Feature Importance — XGBoost (row1, col2-3) ─────────────
ax_fi = fig.add_subplot(gs[1, 2:])
xgb_model = models['XGBoost']
imp = xgb_model.feature_importances_
feat_imp = pd.Series(imp, index=feature_names).sort_values(ascending=True).tail(15)
colors_fi = [ACCENT3 if v > feat_imp.quantile(0.75) else ACCENT1 for v in feat_imp.values]
bars = ax_fi.barh(feat_imp.index, feat_imp.values, color=colors_fi, height=0.65)
ax_fi.set_title("TOP FEATURE IMPORTANCES (XGBoost)", color=ACCENT1, fontsize=11, fontweight='bold', pad=10)
ax_fi.set_xlabel("Importance Score")
ax_fi.grid(True, axis='x')
for bar, val in zip(bars, feat_imp.values):
    ax_fi.text(val + 0.001, bar.get_y() + bar.get_height()/2,
               f'{val:.3f}', va='center', fontsize=7.5, color=TEXT)

# ── 5.6  Churn distribution by Geography (row2, col0) ────────────
ax_geo = fig.add_subplot(gs[2, 0])
geo_cols = [c for c in df.columns if 'Geography' in c]
geo_churn = {c.replace('Geography_',''):
             df[df['Exited']==1][c].sum() / df[c].sum()
             for c in geo_cols}
geos = list(geo_churn.keys())
rates = list(geo_churn.values())
bars2 = ax_geo.bar(geos, rates, color=[ACCENT2, ACCENT4, ACCENT3], width=0.5)
for b, v in zip(bars2, rates):
    ax_geo.text(b.get_x()+b.get_width()/2, v+0.003, f'{v:.1%}',
                ha='center', fontsize=10, fontweight='bold', color=TEXT)
ax_geo.set_title("CHURN RATE BY GEOGRAPHY", color=ACCENT1, fontsize=11, fontweight='bold', pad=10)
ax_geo.set_ylabel("Churn Rate"); ax_geo.set_ylim(0, 0.35)
ax_geo.grid(True, axis='y')

# ── 5.7  Churn by Age Group (row2, col1) ─────────────────────────
ax_age = fig.add_subplot(gs[2, 1])
age_labels = ['18-30','31-45','46-60','60+']
df['AgeGroup_label'] = pd.cut(df['Age'], bins=[0,30,45,60,200], labels=age_labels)
age_churn = df.groupby('AgeGroup_label')['Exited'].mean()
bars3 = ax_age.bar(age_churn.index, age_churn.values,
                   color=[ACCENT1, ACCENT4, ACCENT2, ACCENT3], width=0.5)
for b, v in zip(bars3, age_churn.values):
    ax_age.text(b.get_x()+b.get_width()/2, v+0.003, f'{v:.1%}',
                ha='center', fontsize=10, fontweight='bold', color=TEXT)
ax_age.set_title("CHURN RATE BY AGE GROUP", color=ACCENT1, fontsize=11, fontweight='bold', pad=10)
ax_age.set_ylabel("Churn Rate"); ax_age.set_ylim(0, 0.65)
ax_age.grid(True, axis='y')

# ── 5.8  Churn Probability Distribution (row2, col2-3) ───────────
ax_dist = fig.add_subplot(gs[2, 2:])
best_proba = results[best_name]['proba']
ax_dist.hist(best_proba[y_test==0], bins=50, alpha=0.65,
             color=ACCENT1, label='Retained', density=True)
ax_dist.hist(best_proba[y_test==1], bins=50, alpha=0.65,
             color=ACCENT2, label='Churned', density=True)
ax_dist.axvline(0.5, color=ACCENT4, lw=2, ls='--', label='Threshold 0.5')
ax_dist.set_title(f"CHURN PROBABILITY DISTRIBUTION\n({best_name})",
                  color=ACCENT1, fontsize=11, fontweight='bold', pad=10)
ax_dist.set_xlabel("Predicted Churn Probability")
ax_dist.set_ylabel("Density")
ax_dist.legend(fontsize=9, framealpha=0.2, labelcolor=TEXT)
ax_dist.grid(True)

# ── Watermark ────────────────────────────────────────────────────
fig.text(0.97, 0.01, "Built with XGBoost · Random Forest · GBM · Logistic Regression · SMOTE",
         ha='right', va='bottom', fontsize=7, color=MUTED, style='italic')

plt.savefig('churn_dashboard.png',
            dpi=160, bbox_inches='tight', facecolor=BG)
print("\n✔  Dashboard saved → churn_dashboard.png")



✔  Dashboard saved → churn_dashboard.png


In [10]:
# ─────────────────────────────────────────────
# 6. CLASSIFICATION REPORT
# ─────────────────────────────────────────────
print("\n" + "─"*60)
print(f"  CLASSIFICATION REPORT — {best_name}")
print("─"*60)
print(classification_report(y_test, results[best_name]['preds'],
                             target_names=['Retained','Churned']))

print("\n  FULL METRICS SUMMARY")
print(f"  {'Model':<22}  {'Accuracy':>9}  {'F1':>7}  {'ROC-AUC':>8}  {'PR-AUC':>7}")
print("  " + "─"*56)
for name, res in results.items():
    star = " ★" if name == best_name else ""
    print(f"  {name:<22}  {res['Accuracy']:>9.4f}  {res['F1']:>7.4f}  {res['ROC-AUC']:>8.4f}  {res['PR-AUC']:>7.4f}{star}")

print("\n✔  Pipeline complete!")


────────────────────────────────────────────────────────────
  CLASSIFICATION REPORT — Gradient Boosting
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

    Retained       0.90      0.93      0.91      1593
     Churned       0.67      0.59      0.63       407

    accuracy                           0.86      2000
   macro avg       0.79      0.76      0.77      2000
weighted avg       0.85      0.86      0.86      2000


  FULL METRICS SUMMARY
  Model                    Accuracy       F1   ROC-AUC   PR-AUC
  ────────────────────────────────────────────────────────
  Logistic Regression        0.7820   0.4709    0.7559   0.4863
  Random Forest              0.8200   0.5794    0.8510   0.6449
  Gradient Boosting          0.8590   0.6319    0.8677   0.7089 ★
  XGBoost                    0.8615   0.6262    0.8615   0.7083
  Ensemble                   0.8530   0.6211    0.8656   0.7034

✔  Pipeline complete!
